## Usage example

Here is an example of a LSA pipeline that:
1. Ingests a collection of texts
2. Makes the corresponding document-term matrix using stemming and removing stop words
3. Extracts 40 topics
4. Shows a table with the extracted topics
5. Shows a table with statistical thesaurus entries for selected words  

In [1]:
# Packages used
use ML::LatentSemanticAnalyzer;
use ML::LatentSemanticAnalyzer::Utilities;
use Lingua::EN::Stem::Porter;
use Data::Reshapers;
use Statistics::OutlierIdentifiers;

In [2]:
# Collection of texts
my @dsAbstracts = ML::LatentSemanticAnalyzer::Utilities::get-abstracts-dataset();

# Remove non-strings
@dsAbstracts .= grep({ $_<Abstract> ~~ Str:D });

say "@dsAbstracts.elems : {@dsAbstracts.elems}";
my %docs = @dsAbstracts.map(*<ID>) Z=> @dsAbstracts.map(*<Abstract>);
say "%docs.elems : {%docs.elems}";

@dsAbstracts.elems : 581
%docs.elems : 576


In [3]:
#% html
@dsAbstracts.pick(6)
==> to-html(field-names => <ID Name Title Abstract Track>)

ID,Name,Title,Abstract,Track
2017.Giulio.Alessandrini..Mikayel.Egibyan.Neural.Networks.for.Face.Analysis,Giulio Alessandrini & Mikayel Egibyan,Neural Networks for Face Analysis and OCR,"In this talk, we highlight our recent updates in the fields of optical character recognition (OCR) and face analysis in the Wolfram Language. Using the very latest iteration of both dedicated functions and the machine learning integrated capabilities, we will demonstrate how these applications benefit from increased accuracy and better detection, feature extraction and recognition, with practical examples ranging from face morphing to audio processing. Last, we would like to cover some of our short- to mid-term future plans.",Vizualization/IP
2019.Brett.Champion.Advances.in.Visualization,Brett Champion,Advances in Visualization,"We’ll begin the talk by looking at visualization features and functions that are new in the Wolfram Language in Version 12.0, including displaying data with uncertainty and using gridded displays. Later in the talk, we’ll look forward to upcoming improvements, including great strides in visualizing vector fields.",Uknown
2019.Dan.McDonald.PhD.Ian.Ford.Lin.Cong.and.Peter.Barendse..Ph.D.Automated.Planar.Geometry,"Dan McDonald, PhD, Ian Ford, Lin Cong, and Peter Barendse , Ph.D.",Automated Planar Geometry,"We present updates to the automated geometric functionality of the Wolfram Language introduced in Version 12, including the functions GeometricScene, RandomInstance and FindGeometricConjectures, as well as formally introduce the Version 12.1 function FindGeometricProof. Given a symbolically described, coordinate-free scene in plane geometry, these functions can automatically produce drawings of the scene, conjectures about the scene and human-readable proofs of theorems pertaining to the scene. We also detail progress in bringing this functionality to Wolfram|Alpha, as well as curating theorems from Euclid’s <i>Elements</i> and beyond.",Uknown
2017.Matthew.Green..Rick.Biggerstaff.Authentic.Computational.Thinking.in.a,Matthew Green & Rick Biggerstaff,Authentic Computational Thinking in a Project-Based Public High School,"Seymour Papert gave us the phrase """"Computational Thinking"""" in his 1980 book """"Mindstorms."""" In 1996 he provided further explanation in his paper, """"An Exploration in the Space of Mathematics Educations"""" stating, """"The goal is to use Computational Thinking to forge ideas."""" The contemporary and popular definition of Computational Thinking has been stripped of its power. Computational Thinking as a """"forge for ideas"""" has been replaced with the basic, and low-level techniques of Computer Science. It is our belief that these techniques are important in a deep study of computer science, but, as Stephen Wolfram himself explains, they are not the correct place to start. The Wolfram Language enables Computational Thinking in a profound way: It is a professional tool that is fully accessible to students. Most other tools used to teach Computational Thinking are stripped down, """"toy"""" versions that will each be outgrown. With the Wolfram Language, we have a professional tool that has enabled deep and meaningful computational and mathematical thinking and problem solving among all students at a depth we've only previously imagined possible. We cannot overstate our excitement at the power and potential of the Wolfram Language as a tool to transform all learning. We will present stories of deep student engagement in learning to think with computers and our plans for taking it further in our second year.",Education
2019.Emmanuel.Garces.Geo.Visualization,Emmanuel Garces,Geo Visualization,"Wolfram Language supports different ways of representing geographic data. We basically deal with coordinates, coordinates with associated values and vectors. Here we will survey through visualization functions depending on the type of data and what we want to look at. We will also introduce new geographic vi

In [4]:
# Stemmer function (to preprocess words in the pipeline below)
porter("preparation")

prepar

In [5]:
# Derive stemming rules (to be used in the LSA pipeline)
my %stemming-rules = %docs.values.join(' ').lc.split(/\s | <:punct> /, :skip-empty)>>.trim.unique.map({ $_ => porter($_) });
deduce-type(%stemming-rules)

Assoc(Atom((Str)), Atom((Str)), 6772)

In [6]:
# Words to show statistical thesaurus entries for
sink my @words = <notebook computational function neural talk programming>;

In [7]:
# Reproducible results (within a single session)
sink srand(12);

In [10]:
# Single echo function used in the pipeline
# my &echo-function = {.say for |$_};

# Echoing as a pretty table
sink my &echo-function = {
    say to-pretty-table($_, field-names => [|$_.head.keys.grep(* ~~ / <alpha> .*/).sort, |$_.head.keys.grep(* ~~ /\d+/).sort(*.Num) ], align => 'l')
};

In [11]:
# LSA pipeline
my $lsaObj =
        ML::LatentSemanticAnalyzer.new
                .make-document-term-matrix(docs => %docs, :stop-words, :%stemming-rules, :3min-length)
                .apply-term-weight-functions(
                    global-weight-func => "IDF", 
                    local-weight-func => "None", 
                    normalizer-func => "Cosine"
                )
                .extract-topics(:40number-of-topics, :10min-number-of-documents-per-term, method => "SVD", :60max-steps, tolerance => 1e-4)
                .echo-topics-interpretation(:12number-of-terms, :dataset, :wide-form, :&echo-function)
                .echo-statistical-thesaurus(
                    terms => @words.map({ porter($_) }), 
                    :wide-form, 
                    :12number-of-nearest-neighbors, 
                    method => "cosine", 
                    :&echo-function
                );


+---------------------------------------+------------+------------+------------+----------+-----------+------------+----------+-----------+-----------+-----------+------------+------------+
| Topic                                 | 0          | 1          | 2          | 3        | 4         | 5          | 6        | 7         | 8         | 9         | 10         | 11         |
+---------------------------------------+------------+------------+------------+----------+-----------+------------+----------+-----------+-----------+-----------+------------+------------+
| tpc.000.softwar-develop-cloud         | softwar    | develop    | cloud      | visual   | featur    | applic     | learn    | scienc    | talk      | includ    | educ       | exampl     |
| tpc.001.visual-scienc-featur          | visual     | scienc     | featur     | neural   | learn     | talk       | includ   | exampl    | educ      | function  | audio      | imag       |
| tpc.002.visual-neural-audio           | visual  

LatentSemanticAnalyzer object with 576 documents and 3944 terms.

In [12]:
$lsaObj.echo-document-term-matrix-statistics()

Document-term matrix:
Math::SparseMatrix(:specified-elements(20708), :dimensions((576, 3944)), :density(<5177/567936>))
Number of terms per document:
	min:     0
	mean:    47.642361111111114
	median:  39
	max:     259
	std:     34.975942287086404
Number of documents per term:
	min:     1
	mean:    6.957910750507099
	median:  2
	max:     640
	std:     21.343509288270646


LatentSemanticAnalyzer object with 576 documents and 3944 terms.

---

## Find outliers in the topics interpretation data frame

In [13]:
my @dsTopicsLongForm = |$lsaObj.get-topics-interpretation(:120number-of-terms, :dataset, :!wide-form, :!echo).take-value;
deduce-type(@dsTopicsLongForm)

Vector(Struct([Score, Term, Topic], [Num, Str, Str]), 4800)

In [14]:
#% html
@dsTopicsLongForm.pick(12)
==> to-html(field-names => <Topic Term Score>)

Topic,Term,Score
tpc.004.educ-visual-student,book,0.019932634875272637
tpc.020.label-algorithm-math,asymptot,0.1682167930900236
tpc.008.optim-convex-cloud,cloud,0.44613299783393234
tpc.004.educ-visual-student,platform,0.06518543711057452
tpc.015.technolog-asymptot-repositori,meet,0.025628297108898114
tpc.029.alpha-demonstr-notebook,shape,0.06726667035723022
tpc.038.subject-challeng-human,deriv,0.032368919842040454
tpc.025.entiti-financi-uniti,rule,0.056587328075564464
tpc.010.technolog-optim-convex,onlin,0.045556373598722597
tpc.007.spatial-pattern-anim,examin,0.05058066212497085


In [15]:
# Group by "Topic" and select rows where Score is an outlier according to your function
my @dfTopicsOfOutliersLongForm =
    group-by(@dsTopicsLongForm, "Topic").kv
    .map(-> $t, @g { @g[outlier-identifier(@g.map(*<Score>), identifier => &top-outliers o &quartile-identifier-parameters)] })
    .flat(1);

deduce-type(@dfTopicsOfOutliersLongForm)

Vector(Struct([Score, Term, Topic], [Num, Str, Str]), 817)

In [16]:
#% html
@dfTopicsOfOutliersLongForm.pick(12)
==> to-html(field-names => <Topic Term Score>)

Topic,Term,Score
tpc.029.alpha-demonstr-notebook,introduct,0.1033121307057368
tpc.027.modul-alpha-unit,modul,0.2814729685943793
tpc.009.resourc-repositori-math,overview,0.15311771452317552
tpc.026.modul-imag-dynam,concept,0.17625127469348137
tpc.012.math-resourc-repositori,overview,0.14132588408932584
tpc.036.step-present-rule,decis,0.09096866960787549
tpc.032.analyt-algebra-vector,product,0.12194544918452913
tpc.022.featur-technolog-alpha,physic,0.16661422786187632
tpc.031.support-notebook-dataset,coordin,0.11977853678301598
tpc.032.analyt-algebra-vector,simul,0.10396770805653396


In [33]:
#% html
group-by(@dfTopicsOfOutliersLongForm, "Topic")».elems.sort(*.value).reverse.map({ "{$_.key} : {$_.value}" })
==> to-html(:multi-column, :4columns)

tpc.017.label-featur-cover : 25,tpc.007.spatial-pattern-anim : 23,tpc.039.tool-dai-assur : 21,tpc.009.resourc-repositori-math : 18
tpc.012.math-resourc-repositori : 25,tpc.011.math-audio-uniti : 23,tpc.032.analyt-algebra-vector : 21,tpc.018.audio-technolog-io : 17
tpc.025.entiti-financi-uniti : 25,tpc.021.cloud-introduct-notebook : 23,tpc.029.alpha-demonstr-notebook : 20,tpc.026.modul-imag-dynam : 17
tpc.024.notebook-deploy-cdf : 25,tpc.030.featur-cover-app : 22,tpc.014.uniti-resourc-game : 20,tpc.020.label-algorithm-math : 17
tpc.013.unsupervis-cluster-learn : 25,tpc.031.support-notebook-dataset : 22,tpc.006.optim-convex-softwar : 20,tpc.037.random-effici-statist : 17
tpc.023.audio-compil-dynam : 24,tpc.005.neural-audio-network : 22,tpc.000.softwar-develop-cloud : 19,tpc.035.vector-project-chain : 17
tpc.016.technolog-geometr-neural : 24,tpc.015.technolog-asymptot-repositori : 22,tpc.002.visual-neural-audio : 19,tpc.034.servic-blockchain-introduct : 16
tpc.001.visual-scienc-featur : 24,tpc.010.technolog-optim-convex : 22,tpc.019.librari-compil-perform : 19,tpc.038.subject-challeng-human : 14
tpc.008.optim-convex-cloud : 24,tpc.028.evalu-geometr-algebra : 22,tpc.027.modul-alpha-unit : 18,tpc.036.step-present-rule : 13
tpc.004.educ-visual-student : 24,tpc.022.featur-technolog-alpha : 21,tpc.033.app-deploy-interfac : 18,tpc.003.educ-neural-student : 9


----

## Representation of texts

In [18]:
sink my $query = q:to/END/;
Wolfram notebooks have been copied by other computational systems.
END

In [19]:
$lsaObj.represent-by-terms($query, :!apply-lsi-functions).take-value

Math::SparseMatrix(:specified-elements(4), :dimensions((1, 3944)), :density(<1/986>))

In [20]:
my $m2 = $lsaObj.represent-by-topics($query, :apply-lsi-functions).take-value;
say $m2;
$m2.transpose.print

Math::SparseMatrix(:specified-elements(40), :dimensions((1, 40)), :density(1.0))
–––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––
                                        id.0                 
––––––––––––––––––––––––––––––––––––––┼––––––––––––––––––––––
tpc.000.softwar-develop-cloud         │ 0.10574422952817113  
tpc.001.visual-scienc-featur          │ 0.03828717113222648  
tpc.002.visual-neural-audio           │ 0.003967319997671845 
tpc.003.educ-neural-student           │ 0.0355569957323533   
tpc.004.educ-visual-student           │ -0.003197932662175004
tpc.005.neural-audio-network          │ -0.060302456602164344
tpc.006.optim-convex-softwar          │ -0.15461502203779542 
tpc.007.spatial-pattern-anim          │ -0.006956674045907262
tpc.008.optim-convex-cloud            │ 0.028760865129505486 
tpc.009.resourc-repositori-math       │ 0.0496770036695354   
tpc.010.technolog-optim-convex        │ -0.04354303812965374 
tpc.011.math-audio-uniti              │ 0.036569616

----

## Profiling

In [21]:
# LSA pipeline
my $lsaObj = ML::LatentSemanticAnalyzer.new;

LatentSemanticAnalyzer object.

In [22]:
$lsaObj .= make-document-term-matrix(docs => %docs, :stop-words, :%stemming-rules, :3min-length);

LatentSemanticAnalyzer object with 576 documents and 3944 terms.

In [23]:
$lsaObj .= apply-term-weight-functions(
                global-weight-func => "IDF", 
                local-weight-func => "None", 
                normalizer-func => "Cosine"
           )        

LatentSemanticAnalyzer object with 576 documents and 3944 terms.

In [24]:
$lsaObj .= extract-topics(:40number-of-topics, :10min-number-of-documents-per-term, method => "SVD", :60max-steps, tolerance => 1e-4)

LatentSemanticAnalyzer object with 576 documents and 3944 terms.

In [25]:
$lsaObj .= echo-topics-interpretation(:12number-of-terms, :dataset, :wide-form, :&echo-function)

+---------------------------------------+------------+------------+------------+----------+-----------+------------+----------+-----------+-----------+-----------+------------+------------+
| Topic                                 | 0          | 1          | 2          | 3        | 4         | 5          | 6        | 7         | 8         | 9         | 10         | 11         |
+---------------------------------------+------------+------------+------------+----------+-----------+------------+----------+-----------+-----------+-----------+------------+------------+
| tpc.000.softwar-develop-cloud         | softwar    | develop    | cloud      | visual   | featur    | applic     | learn    | scienc    | talk      | includ    | educ       | exampl     |
| tpc.001.visual-scienc-featur          | visual     | scienc     | featur     | neural   | learn     | talk       | includ   | exampl    | educ      | function  | audio      | imag       |
| tpc.002.visual-neural-audio           | visual  

LatentSemanticAnalyzer object with 576 documents and 3944 terms.

In [26]:
$lsaObj .= echo-statistical-thesaurus(
                terms => @words.map({ porter($_) }), 
                :wide-form, 
                :12number-of-nearest-neighbors, 
                method => "cosine", 
                :&echo-function
            );

+------------+-----------------------------------------------------------------------------------------------+
| SearchTerm | Terms                                                                                         |
+------------+-----------------------------------------------------------------------------------------------+
| comput     | comput idea think languag activ integr audienc depth transform discuss experi extens          |
| function   | function includ introduc talk languag domain futur type special discuss version term          |
| neural     | neural network train sequenc framework achiev deep classif task workshop workflow guid        |
| notebook   | notebook introduct edit alpha select hand initi cloud singl cdf book colleg                   |
| program    | program code mathematica requir detail base manag rich skill knowledg languag receiv          |
| talk       | talk languag function includ domain analysi object provid discuss demonstr mathematica applic |
+

LatentSemanticAnalyzer object with 576 documents and 3944 terms.